# Stage 2 -- Aggregation: Daily Means

## Input
- `Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_stock_daily_engineered.parquet` -- Panel A, ~100 stocks × ~4,656 dates × 192 factors, keyed on `(permno, date)`
- `Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_macro_daily_engineered.parquet` -- Panel C, market-level macro daily factors, keyed on `date`

## Purpose
Transforms Panel A (stock-level daily) into a single market-level daily time series by computing cap-weighted means cross-sectionally per date, then merges with Panel C (macro daily) to produce the daily means feature table. The output is a flat daily table keyed on `date` suitable for model training.

---

## Pipeline

### Step 1: Load & Trim
Both panels are loaded and Panel A is trimmed to dates on or after 2006-07-03, aligning with Panel C which starts July 2006 due to CFTC data availability. Date alignment between Panel A and Panel C is verified (common dates, dates only in A, dates only in C). Factor column lists are identified separately for stock and macro panels.

### Step 2: Handle Warmup NaN
NaN rates per factor are reported. Rolling features from Panel A (momentum windows, volatility dynamics, etc.) have NaN in early rows due to warmup periods. **Strategy: leave NaN in place.** The cap-weighted aggregation in Step 5 computes each factor's mean using only stocks with valid (non-NaN) values on that date, excluding NaN stocks per factor per date naturally.

### Step 3: Winsorise
All stock factor columns are cast to `float64`. Cross-sectional winsorisation is applied at the 1st/99th percentile per date using a fully vectorised approach: `groupby('date')[col].transform('quantile', 0.01/0.99)` followed by `clip()`. This stays in the pandas C-engine with no Python loops over dates.

**Post-winsorisation drop:** Three ISO (intermarket sweep order) columns are dropped -- `iso_dollar_to_cap`, `iso_vol_to_shrout`, `n_iso_trade_pct` -- because their data starts 162 days late and they are marginal given other order flow coverage already in the panel.

### Step 4: Compute Target Variable
The target is the **next-day cap-weighted market return**: for each date t, the target = Σ(cap_{i,t} × ret_{i,t+1}) / Σ(cap_{i,t}), using today's market cap weights and tomorrow's total returns.

Implementation:
- Panel A is sorted by `(permno, date)`
- `next_day_ret` = `dlyret` shifted back by 1 within each PERMNO
- A date-gap guard nulls out `next_day_ret` where consecutive dates are more than 5 days apart (prevents year-boundary or gap-boundary errors)
- Weighted returns and total cap are summed per date to produce `target_daily_return`

The last trading day correctly has no target (no t+1 return available). Target statistics (mean, std, min, max, NaN count) are reported.

### Step 5: Cap-Weighted Mean Aggregation
For each of the ~189 surviving stock factor columns, the cap-weighted mean is computed per date using a vectorised approach:

- Factor values and cap weights are extracted as numpy arrays
- Valid mask: non-NaN in both cap and factor
- Weighted values and valid caps are computed element-wise
- `groupby('date').sum()` is called once per factor (pandas C-engine)
- cwmean = weighted sum / cap sum (NaN where cap sum is zero)

Results are assembled into a single DataFrame in one shot to avoid DataFrame fragmentation. NaN counts in the aggregated output are reported.

### Step 6: Merge Aggregated Stock + Macro Daily + Target
Column name conflicts between aggregated stock factors and Panel C macro factors are checked and resolved with a `stock_` prefix if needed. Three-way merge on `date`:
1. Aggregated stock cwmeans (inner join with Panel C)
2. Panel C macro factors
3. Target (`target_daily_return`, left join)

**Warmup trim:** The first 50 rows are dropped after merging to allow Panel C's longest rolling window (50-day MA) to warm up. The last row (no target available) is also dropped.

### Step 7: Validation
- No duplicate dates
- NaN count across all feature columns (reported per column if any remain)
- Target NaN count (expect 0 after dropping the last row)
- **Target integrity check:** correlation between `target_daily_return` on date t and the cwmean of `dlyretx` on date t+1 (should be ~0.99+, not exactly 1.0 due to dividends and index composition changes)
- Row count and dates lost in the inner merge
- Column breakdown: stock cwmean factors, macro factors, target, date

### Step 8: Save
Sorted by date and saved to parquet.

---

## Key Design Decisions
- **Cap-weighted mean only.** Higher moments (std, skew, kurt, spread) are computed in a separate notebook (`02_build_agg_daily_full_moments.ipynb`) to keep this notebook fast and focused.
- **Inner join on date** between stock aggregation and Panel C. Dates present in one but not the other are dropped.
- **Vectorised winsorisation and aggregation** -- no Python loops over dates, keeping runtime manageable for ~525K stock-day rows.
- **ISO columns dropped post-winsorisation** rather than pre, so the winsorisation summary reflects the full pre-drop factor set.
- **50-row warmup trim** applied after merge to handle Panel C rolling features (VIX vs MA50 has the longest warmup).

## Output
`Data/Data_Collection/Final/Stage_2/agg_market_daily_means.parquet` -- keyed on `date`, containing cap-weighted mean of each stock daily factor plus all Panel C macro factors and `target_daily_return`

In [6]:
# %% [markdown]
# # Stage 2 — Aggregation: Daily Means
#
# Transforms Panel A (stock daily, ~100 stocks × ~4,656 dates × 192 factors)
# into a single market-level daily time series using cap-weighted means.
# Merges with Panel C (macro daily) to produce the daily means feature table.
#
# Pipeline (order is critical):
#   1. Load & trim to ≥ 2006-07-03
#   2. Handle warmup NaN from rolling features
#   3. Winsorise stock factors at 1st/99th cross-sectionally per date
#   4. Compute target: next-day cap-weighted market return
#   5. Aggregate: cap-weighted mean per date
#   6. Merge aggregated stock means + macro daily on date
#   7. Validate: zero NaN, target integrity, date alignment
#   8. Save
#
# Input:
#   Panel A: Stage_1_5/.../panel_stock_daily_engineered.parquet
#   Panel C: Stage_1_5/.../panel_macro_daily_engineered.parquet
#
# Output:
#   Stage_2/agg_market_daily_means.parquet

# %%
import pandas as pd
import numpy as np
from pathlib import Path
import time

PANEL_A_PATH = Path('../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_stock_daily_engineered.parquet')
PANEL_C_PATH = Path('../../../Data/Data_Collection/Final/Stage_1_5_Validation_and_Feature_Engineering/panel_macro_daily_engineered.parquet')
OUT_DIR = Path('../../../Data/Data_Collection/Final/Stage_2')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 1: LOAD & TRIM
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("=" * 90)
print("STEP 1: LOAD & TRIM")
print("=" * 90)

START_DATE = '2004-01-02'

# Minimum stocks with data for a cap-weighted statistic to be reported.
# A cwmean over 15 of ~100 stocks is not the market average -- it is the average
# among whichever stocks happened to be covered, and that composition drifts
# systematically (BATS: 66% NaN in 2004 falling to 0.9% by 2020). A stock count
# is needed rather than a date cutoff because BATS coverage improves gradually
# across fifteen years and never becomes complete.
MIN_STOCKS = 50

# Load Panel A (stock daily)
panel_a = pd.read_parquet(PANEL_A_PATH)
panel_a['date'] = pd.to_datetime(panel_a['date'])
print(f"\n  Panel A loaded: {panel_a.shape[0]:,} rows × {panel_a.shape[1]} columns")
print(f"    Date range: {panel_a['date'].min().date()} → {panel_a['date'].max().date()}")

# Trim to post June 2006
panel_a = panel_a[panel_a['date'] >= START_DATE].reset_index(drop=True)
print(f"    After trim:  {panel_a.shape[0]:,} rows")
print(f"    Date range: {panel_a['date'].min().date()} → {panel_a['date'].max().date()}")
print(f"    Unique dates: {panel_a['date'].nunique():,}")
print(f"    Avg stocks/date: {panel_a.groupby('date').size().mean():.1f}")

# Load Panel C (macro daily)
panel_c = pd.read_parquet(PANEL_C_PATH)
panel_c['date'] = pd.to_datetime(panel_c['date'])
print(f"\n  Panel C loaded: {panel_c.shape[0]:,} rows × {panel_c.shape[1]} columns")
print(f"    Date range: {panel_c['date'].min().date()} → {panel_c['date'].max().date()}")

# Identify columns
meta_cols = ['permno', 'date', 'dlyret', 'dlycap']
factor_cols = [c for c in panel_a.columns if c not in meta_cols]
macro_factor_cols = [c for c in panel_c.columns if c != 'date']

print(f"\n  Panel A factor columns: {len(factor_cols)}")
print(f"  Panel C factor columns: {len(macro_factor_cols)}")

# Verify date alignment
a_dates = set(panel_a['date'].unique())
c_dates = set(panel_c['date'].unique())
common_dates = a_dates & c_dates
only_a = a_dates - c_dates
only_c = c_dates - a_dates

print(f"\n  Date alignment:")
print(f"    Common dates: {len(common_dates):,}")
print(f"    Only in Panel A: {len(only_a)}")
print(f"    Only in Panel C: {len(only_c)}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 2: HANDLE WARMUP NaN
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 2: HANDLE WARMUP NaN FROM ROLLING FEATURES")
print("=" * 90)

# Count NaN per factor
nan_before = panel_a[factor_cols].isna().sum()
nan_factors = nan_before[nan_before > 0]
print(f"\n  Factors with NaN: {len(nan_factors)} / {len(factor_cols)}")
print(f"  Total NaN cells: {nan_before.sum():,}")

if len(nan_factors) > 0:
    print(f"\n  Top 10 by NaN count:")
    for c in nan_factors.sort_values(ascending=False).head(10).index:
        n = int(nan_factors[c])
        pct = n / len(panel_a) * 100
        print(f"    {c:<40s} {n:>7,d} ({pct:.2f}%)")

print(f"\n  Strategy: leave NaN in place. Cap-weighted aggregation excludes")
print(f"  NaN stocks from each factor's computation (per-factor, per-date).")

# ══════════════════════════════════════════════════════════════════════
# STEP 3: WINSORISE — FULLY VECTORISED (no Python loops)
# ══════════════════════════════════════════════════════════════════════

t0 = time.time()

for c in ['permno', 'date', 'dlyret', 'dlycap']:
    assert c not in factor_cols, f"FATAL: {c} is in factor_cols!"

# Cast all factor columns to float64 (fixes nullable Int64/Float64 issues)
panel_a[factor_cols] = panel_a[factor_cols].astype('float64')


for col in factor_cols:
    p01 = panel_a.groupby('date')[col].transform('quantile', 0.01)
    p99 = panel_a.groupby('date')[col].transform('quantile', 0.99)
    panel_a[col] = panel_a[col].clip(lower=p01, upper=p99)


elapsed = time.time() - t0
total_cells = len(panel_a) * len(factor_cols)
print(f"\n  Winsorised {len(factor_cols)} factors in {elapsed:.1f}s")
print(f"  Total factor cells: {total_cells:,}")
print(f"  ✓ Winsorisation complete")

# Drop ISO columns — data starts 162 days late, marginal given other order flow coverage
iso_drop = ['iso_dollar_to_cap', 'iso_vol_to_shrout', 'n_iso_trade_pct']
iso_drop = [c for c in iso_drop if c in factor_cols]
panel_a = panel_a.drop(columns=iso_drop)
factor_cols = [c for c in factor_cols if c not in iso_drop]
print(f"Dropped {len(iso_drop)} ISO columns (late-starting, redundant with other order flow)")


#####
####
###
#

# Per-factor, per-year: % of trading days with fewer than MIN_STOCKS valid
# observations -- i.e. days the gate turns that factor into NaN
cap_ok = panel_a['dlycap'].notna().to_numpy()
valid = panel_a[factor_cols].notna() & cap_ok[:, None]

fail = valid.groupby(panel_a['date']).sum() < MIN_STOCKS
pct = (fail.groupby(fail.index.year).mean() * 100).T
pct = pct.loc[pct.max(axis=1) > 0]

print(f"MIN_STOCKS = {MIN_STOCKS}   "
      f"({len(pct)} of {len(factor_cols)} factors fail on at least one day)\n")
print(pct.round(0).astype(int).to_string())


# ═══════════════════════════════════════════════════════════════════════════════
# STEP 4: COMPUTE TARGET VARIABLE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 4: COMPUTE TARGET (next-day cap-weighted market return)")
print("=" * 90)

# Target = Σ(cap_{i,t} × ret_{i,t+1}) / Σ(cap_{i,t})
# Uses today's weights (known at end of day t) × tomorrow's returns.

panel_a = panel_a.sort_values(['permno', 'date']).reset_index(drop=True)

# Next-day return per stock
panel_a['next_day_ret'] = panel_a.groupby('permno')['dlyret'].shift(-1)

# Guard: null out where consecutive dates >5 days apart (year/gap boundaries)
date_diff = panel_a.groupby('permno')['date'].diff(-1).abs()
panel_a.loc[date_diff > pd.Timedelta(days=5), 'next_day_ret'] = np.nan

# Aggregate to market-level target per date
target_df = panel_a.dropna(subset=['next_day_ret', 'dlycap']).copy()
target_df['weighted_next_ret'] = target_df['dlycap'] * target_df['next_day_ret']

target_agg = target_df.groupby('date').agg(
    target_daily_return=('weighted_next_ret', 'sum'),
    total_cap=('dlycap', 'sum'),
).reset_index()
target_agg['target_daily_return'] = target_agg['target_daily_return'] / target_agg['total_cap']
target_agg = target_agg[['date', 'target_daily_return']]

print(f"\n  Target computed: {len(target_agg):,} dates")
print(f"    Mean: {target_agg['target_daily_return'].mean():.6f}")
print(f"    Std:  {target_agg['target_daily_return'].std():.6f}")
print(f"    Min:  {target_agg['target_daily_return'].min():.6f}")
print(f"    Max:  {target_agg['target_daily_return'].max():.6f}")
print(f"    NaN:  {target_agg['target_daily_return'].isna().sum()}")

# Last trading day should have NaN target
last_date = panel_a['date'].max()
last_target = target_agg[target_agg['date'] == last_date]['target_daily_return']
if len(last_target) == 0 or last_target.isna().all():
    print(f"  ✓ Last date ({last_date.date()}) correctly has no target")
else:
    print(f"  ⚠ Last date has target = {last_target.values[0]:.6f} — check if correct")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 5: CAP-WEIGHTED MEAN AGGREGATION
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 5: CAP-WEIGHTED MEAN AGGREGATION")
print("=" * 90)

t0 = time.time()

# VECTORISED APPROACH: for each factor, compute weighted sum and weight sum
# using groupby().sum() (C-optimised in pandas), then divide.
# Much faster than groupby().apply() with a Python function.

agg_factors = factor_cols.copy()
cap_arr = panel_a['dlycap'].to_numpy(dtype='float64', na_value=np.nan)

# Get sorted unique dates for the result index
sorted_dates = np.sort(panel_a['date'].unique())

print(f"\n  Aggregating {len(agg_factors)} factors across {len(sorted_dates):,} dates...")

agg_results = {}
for i, col in enumerate(agg_factors):
    vals = panel_a[col].to_numpy(dtype='float64', na_value=np.nan)
    
    # Valid where both cap and factor are non-NaN
    valid = ~(np.isnan(vals) | np.isnan(cap_arr))
    
    # Weighted values and valid caps
    weighted = np.where(valid, cap_arr * vals, 0.0)
    cap_valid = np.where(valid, cap_arr, 0.0)
    
    # Groupby sum (pandas C-engine, very fast)
    temp = pd.DataFrame({
        'date': panel_a['date'].values,
        'wv': weighted,
        'wc': cap_valid,
        'n': valid.astype('int64'),
    })
    agg = temp.groupby('date', sort=True).sum()
    cwmean = (agg['wv'] / agg['wc'].replace(0, np.nan)).values
    # Suppress dates where too few stocks have data (see MIN_STOCKS above)
    agg_results[col] = np.where(agg['n'].values >= MIN_STOCKS, cwmean, np.nan)
    
    if (i + 1) % 50 == 0:
        print(f"    {i + 1}/{len(agg_factors)} factors done...")

# Build result DataFrame in one shot (avoids fragmentation warning)
agg_results['date'] = sorted_dates
agg_stock = pd.DataFrame(agg_results)

elapsed = time.time() - t0
print(f"\n  Aggregated in {elapsed:.1f}s")
print(f"  Result: {agg_stock.shape[0]:,} rows × {agg_stock.shape[1]} columns")

# Check for NaN in aggregated data
agg_nan = agg_stock[agg_factors].isna().sum()
agg_nan_cols = agg_nan[agg_nan > 0]
if len(agg_nan_cols) > 0:
    print(f"\n  Factors with NaN after aggregation: {len(agg_nan_cols)}")
    for c in agg_nan_cols.sort_values(ascending=False).head(25).index:
        n = int(agg_nan_cols[c])
        print(f"    {c:<45s} {n:>5d} NaN ({n / len(agg_stock) * 100:5.1f}%)")

    # Flag anything MIN_STOCKS may have gutted rather than merely trimmed
    gutted = agg_nan_cols[agg_nan_cols > 0.5 * len(agg_stock)]
    if len(gutted):
        print(f"\n  ** {len(gutted)} factors NaN on >50% of dates -- check "
              f"whether MIN_STOCKS={MIN_STOCKS} is too high for these **")
        for c in gutted.index:
            print(f"    {c}")
else:
    print(f"  ✓ Zero NaN in aggregated stock factors")




# ═══════════════════════════════════════════════════════════════════════════════
# STEP 6: MERGE WITH MACRO DAILY + TARGET
# ═══════════════════════════════════════════════════════════════════════════════


# %%
print("\n" + "=" * 90)
print("STEP 6: MERGE AGGREGATED STOCK + MACRO DAILY + TARGET")
print("=" * 90)

# Check for column name conflicts
stock_cols = set(agg_stock.columns) - {'date'}
macro_cols = set(panel_c.columns) - {'date'}
overlap = stock_cols & macro_cols
if overlap:
    print(f"\n  ⚠ Column name conflicts ({len(overlap)}):")
    for c in sorted(overlap):
        print(f"    {c}")
    print(f"    Adding 'stock_' prefix to conflicting stock columns...")
    rename_map = {c: f'stock_{c}' for c in overlap}
    agg_stock = agg_stock.rename(columns=rename_map)
    agg_factors = [rename_map.get(c, c) for c in agg_factors]
else:
    print(f"\n  ✓ No column name conflicts between stock and macro")

# Merge: stock agg + macro + target (all on date)
result = agg_stock.merge(panel_c, on='date', how='inner')
result = result.merge(target_agg, on='date', how='left')

# Trim warmup rows (longest rolling window in Panel C = 50 days)
pre_trim = len(result)
result = result.iloc[50:].reset_index(drop=True)
print(f"Trimmed first 50 rows for warmup: {pre_trim} → {len(result)}")

# Drop last row (no target — can't train on it)
result = result.dropna(subset=['target_daily_return']).reset_index(drop=True)
print(f"  Dropped last row (no next-day return): {len(result):,} rows")

print(f"\n  Merged result: {result.shape[0]:,} rows × {result.shape[1]} columns")
print(f"  Date range: {result['date'].min().date()} → {result['date'].max().date()}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 7: VALIDATE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 7: VALIDATE")
print("=" * 90)

# 7a. Duplicate dates
n_dupes = result['date'].duplicated().sum()
print(f"\n  Duplicate dates: {n_dupes}")
assert n_dupes == 0, "FATAL: Duplicate dates in result!"
print(f"  ✓ No duplicate dates")

# 7b. NaN in features
feature_cols = [c for c in result.columns if c not in ['date', 'target_daily_return']]
feature_nan = result[feature_cols].isna().sum()
feature_nan_total = feature_nan.sum()
print(f"\n  Feature NaN: {feature_nan_total}")

if feature_nan_total > 0:
    nan_cols = feature_nan[feature_nan > 0].sort_values(ascending=False)
    print(f"  Columns with NaN ({len(nan_cols)}):")
    for c in nan_cols.head(15).index:
        print(f"    {c}: {int(nan_cols[c])}")
else:
    print(f"  ✓ Zero NaN in features")


# ── 7b2. Trailing NaN ───────────────────────────────────────────────────────
# A factor that stops publishing leaves the END of the sample empty, which lands
# in the test period. Split_D tests 2023-2024. The 5 discontinued OAP factors are
# dropped by name in notebooks 03/04, so anything listed here is new.
TRAILING_TOLERANCE = 126        # ~6 months of trading days

n_rows = len(result)
stopped = {}
for c in feature_cols:
    lv = result[c].last_valid_index()
    if lv is None:
        stopped[c] = ('never valid', n_rows)
    elif n_rows - 1 - lv > TRAILING_TOLERANCE:
        stopped[c] = (str(result['date'].iloc[lv].date()), n_rows - 1 - lv)

if stopped:
    print(f"\n  ** {len(stopped)} columns stop >{TRAILING_TOLERANCE} rows "
          f"before {result['date'].max().date()} **")
    print(f"  {'Column':<45s} {'Last valid':>12s} {'Rows missing':>13s}")
    for c, (d, g) in sorted(stopped.items(), key=lambda x: -x[1][1]):
        print(f"  {c:<45s} {d:>12s} {g:>13d}")
else:
    print(f"  ✓ No columns stop early")

# 7c. Target NaN — should only be last date
target_nan = result['target_daily_return'].isna().sum()
print(f"\n  Target NaN: {target_nan} (expect 1 — last trading day)")

# 7d. Target integrity
# The target on date t should approximately equal the cwmean of dlyretx on date t+1
if 'dlyretx' in result.columns:
    today_ret_col = 'dlyretx'
elif 'stock_dlyretx' in result.columns:
    today_ret_col = 'stock_dlyretx'
else:
    today_ret_col = None

if today_ret_col:
    result_sorted = result.sort_values('date')
    shifted_mkt = result_sorted[today_ret_col].shift(-1)
    corr = result_sorted['target_daily_return'].corr(shifted_mkt)
    print(f"\n  Target integrity: corr(target_t, cwmean_dlyretx_{today_ret_col}_t+1) = {corr:.6f}")
    print(f"    (Should be ~0.99+, not exactly 1.0 due to dividends + composition changes)")
    if corr < 0.95:
        print(f"  ⚠ Correlation lower than expected — investigate!")
    else:
        print(f"  ✓ Target correctly represents next-day cap-weighted return")

# 7e. Row count
print(f"\n  Row count: {len(result):,}")
print(f"  Panel C rows: {len(panel_c):,}")
print(f"  Dates lost in merge: {len(panel_c) - len(result)}")

# 7f. Column breakdown
stock_feature_count = len([c for c in result.columns if c in agg_factors])
macro_feature_count = len([c for c in result.columns if c in macro_factor_cols])
print(f"\n  Column breakdown:")
print(f"    Stock cwmean factors: {stock_feature_count}")
print(f"    Macro factors:        {macro_feature_count}")
print(f"    Target:               1")
print(f"    Date:                 1")
print(f"    Total:                {result.shape[1]}")

# ═══════════════════════════════════════════════════════════════════════════════
# STEP 8: SAVE
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("STEP 8: SAVE")
print("=" * 90)

result = result.sort_values('date').reset_index(drop=True)

out_path = OUT_DIR / 'agg_market_daily_means.parquet'
result.to_parquet(out_path, index=False, engine='pyarrow')

file_size = out_path.stat().st_size
print(f"\n  ✓ Saved: {out_path}")
print(f"    {result.shape[0]:,} rows × {result.shape[1]} columns")
print(f"    Size: {file_size / 1e6:.1f} MB")

# ═══════════════════════════════════════════════════════════════════════════════
# FINAL SUMMARY
# ═══════════════════════════════════════════════════════════════════════════════

# %%
print("\n" + "=" * 90)
print("DAILY MEANS AGGREGATION COMPLETE")
print("=" * 90)

print(f"""
  Pipeline:
    Panel A ({panel_a.shape[0]:,} stock-days) → winsorise → cwmean → {stock_feature_count} factors
    Panel C ({len(panel_c):,} days) → {macro_feature_count} factors
    Target: next-day cap-weighted market return (from stock-level shift)
    Merge: inner join on date

  Result:
    Rows:    {result.shape[0]:,} trading days
    Columns: {result.shape[1]} ({stock_feature_count} stock + {macro_feature_count} macro + target + date)
    Dates:   {result['date'].min().date()} → {result['date'].max().date()}
    NaN:     {feature_nan_total} features + {target_nan} target (last day only)

  Saved: {out_path}

  Next: 02_build_agg_daily_full_moments.ipynb (adds cwstd, cwskew, cwkurt, spread)
""")

STEP 1: LOAD & TRIM

  Panel A loaded: 525,957 rows × 196 columns
    Date range: 2004-01-02 → 2024-12-31
    After trim:  525,957 rows
    Date range: 2004-01-02 → 2024-12-31
    Unique dates: 5,285
    Avg stocks/date: 99.5

  Panel C loaded: 5,285 rows × 210 columns
    Date range: 2004-01-02 → 2024-12-31

  Panel A factor columns: 192
  Panel C factor columns: 209

  Date alignment:
    Common dates: 5,285
    Only in Panel A: 0
    Only in Panel C: 0

STEP 2: HANDLE WARMUP NaN FROM ROLLING FEATURES

  Factors with NaN: 187 / 192
  Total NaN cells: 4,126,860

  Top 10 by NaN count:
    n_iso_trade_pct                           89,907 (17.09%)
    iso_vol_to_shrout                         89,904 (17.09%)
    iso_dollar_to_cap                         89,904 (17.09%)
    total_vol_b_to_shrout                     80,032 (15.22%)
    total_dollar_b_to_cap                     80,032 (15.22%)
    total_n_trades_b_pct                      80,032 (15.22%)
    venue_range_b                  